# 06 — Real data experiments (section 6.2 and appendix C.1)


## Setup


In [ ]:
# --- make src/ importable: src/common + the experiment folder --------------
import os, sys
from pathlib import Path
_ROOT = Path.cwd()
while not (_ROOT / "src").is_dir() and _ROOT != _ROOT.parent:
    _ROOT = _ROOT.parent
sys.path[:0] = [str(_ROOT / "src" / "common"), str(_ROOT / "src" / "reel_graphs")]

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from joblib import Parallel, delayed

import graph_learning as gl
import metrics as mt
import paths
import algorithms as al            # src/reel_graphs/algorithms.py
import build_reel_graphs as brg    # src/reel_graphs/build_reel_graphs.py
import interim_calcul as ic        # src/reel_graphs/interim_calcul.py
import plot_reel_graphs as prg     # src/reel_graphs/plot_reel_graphs.py

EPS        = al.EPS_DEFAULT
MAXIT_LOG  = al.MAX_ITER_DEFAULT
MAXIT_DONG = al.MAX_ITER_DEFAULT
N_RUNS     = 100            # k-means runs and label subsets, as in the paper
# grids calibrated on USPS so the densities land on the paper's axis, 3 to 22
import run_experiments_reel_graphs as rer
BETAS      = rer.GRIDS["USPS"]["betas"]     # model (12)
ALPHAS     = rer.GRIDS["USPS"]["alphas"]    # model (14)
KS         = rer.GRIDS["USPS"]["ks"]
COLORS     = prg.COLORS

USE_TORCH  = False          # True runs the solvers through torch_backend (CUDA)
print("cores:", os.cpu_count(), "| repo root:", _ROOT)


## Data


In [ ]:
USPS_X,  _usps_y = ic.load_dataset("USPS")
USPS_y  = np.array([("10" if c == 0 else str(c)) for c in _usps_y])

MNIST_X, _mnist_y = ic.load_dataset("MNIST")
MNIST_y = _mnist_y.astype(str)

COIL_X,  _coil_y = ic.load_dataset("COIL20")
COIL_y  = _coil_y.astype(str)

MNIST_X = MNIST_X / 255.0

for n, X, y in [("USPS", USPS_X, USPS_y), ("MNIST", MNIST_X, MNIST_y),
                ("COIL-20", COIL_X, COIL_y)]:
    print(f"{n:8s} {str(X.shape):14s} {len(np.unique(y)):3d} classes  "
          f"range [{X.min():.2f}, {X.max():.2f}]")


### MNIST distance check


In [ ]:
for dig in ["1", "2"]:
    A = MNIST_X[MNIST_y == dig]
    s = (A ** 2).sum(1)
    D = s[:, None] + s[None, :] - 2 * A @ A.T
    print(f'digit "{dig}": mean square distance = {D[np.triu_indices(len(A), 1)].mean():6.2f}'
          f'   (paper: {45 if dig == "1" else 102})')


## Protocol


In [ ]:
def learn(z, m, model, params, max_iter, n_jobs=-1):
    if USE_TORCH:
        import torch_backend as tb
        first  = params if model == "dong" else [1.0] * len(params)
        second = [float(m)] * len(params) if model == "dong" else params
        Wb, _, its = tb.fbf_batch(z, first, second, m, model=model, eps=EPS,
                                  max_iter=max_iter)
        return [(float(p), Wb[k], int(its[k]), its[k] < max_iter)
                for k, p in enumerate(params)]
    return al.learn_grid(z, m, model, params, max_iter=max_iter, eps=EPS, n_jobs=n_jobs)


def score_w(w, m, labels, label_rate, n_runs=N_RUNS, cluster=True, rng=0):
    return al.score_weights(w, m, labels, label_rate=label_rate, n_runs=n_runs,
                            cluster=cluster, seed=rng)


def sweep(X, labels, label_rate, cluster=True, n_runs=N_RUNS, rng=0, keep_w=False,
          betas=BETAS, alphas=ALPHAS, ks=KS, verbose=True, n_jobs=-1):
    m = X.shape[0]
    z = brg.normalized_dist_vector(X)
    records, store = al.sweep_models(z, m, labels, betas, alphas, ks,
                                     label_rate=label_rate, n_runs=n_runs,
                                     cluster=cluster, max_iter=MAXIT_LOG, eps=EPS,
                                     n_jobs=n_jobs, seed=rng, keep_weights=keep_w,
                                     verbose=verbose)
    return pd.DataFrame(records), z, store


def curve(ax, df, col, dash=None, xmax=22):
    for name in ["Dong etal", "Ours", "k-NN"]:
        sub = df[(df.model == name) & (df.density <= xmax)].sort_values("density")
        if not len(sub):
            continue
        ax.plot(sub.density, sub[col], "-o", ms=3, color=COLORS[name], label=name)
        if dash is not None:
            ax.plot(sub.density, sub[dash], "--", lw=1.2, color=COLORS[name])
    ax.set_xlabel("graph density"); ax.grid(alpha=0.3)


def pick(df, store, model, target):
    sub = df[df.model == model]
    row = sub.iloc[int((sub.density - target).abs().argmin())]
    return row, store[(model, float(row.param))]


def draw(ax, w, coords, values, title, cmap="tab10", vmin=None, vmax=None,
         rel_tol=1e-6, s=16, edge_lw=0.3, edge_alpha=0.35):
    return ic.display_graph_scatter_interim(ax, w, coords, values, title, cmap=cmap,
                                            rel_tol=rel_tol, vmin=vmin, vmax=vmax,
                                            edge_alpha=edge_alpha, node_size=s)


## USPS


In [ ]:
rng = np.random.default_rng(0)
classes = sorted(np.unique(USPS_y).tolist(), key=int)
SIZES = [round(2.6 * i ** 2) for i in range(1, 11)]
sel = np.concatenate([rng.choice(np.flatnonzero(USPS_y == c), s, replace=False)
                      for c, s in zip(classes, SIZES)])
Xu, yu = USPS_X[sel], USPS_y[sel]
print("class sizes:", SIZES, " total m =", sum(SIZES))


In [ ]:
t0 = time.perf_counter()
DF_USPS, Z_USPS, W_USPS = sweep(Xu, yu, label_rate=0.10, keep_w=True)
print(f"total {time.perf_counter()-t0:.0f}s")
print("not converged:",
      DF_USPS[~DF_USPS.converged][["model", "param", "n_iter", "density"]].to_string(index=False)
      if (~DF_USPS.converged).any() else "none")
DF_USPS.round(3)


### Figure 1


In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 4.4))

curve(ax[0], DF_USPS, "clustering_error")
ax[0].set_ylabel("clustering error"); ax[0].set_title("Spectral Clustering")
ax[0].legend(fontsize=9)

curve(ax[1], DF_USPS, "lp_miss")
ax[1].set_ylabel("classification error"); ax[1].set_title("Label Propagation")

curve(ax[2], DF_USPS, "n_components")
sub = DF_USPS[(DF_USPS.model == "Dong etal") & (DF_USPS.density <= 22)].sort_values("density")
ax[2].plot(sub.density, sub.n_disconnected_nodes, "--", color=COLORS["Dong etal"],
           label="# disconnected (Dong etal)")
ax[2].axhline(10, color="k", lw=1, ls=":", label="number of classes")
ax[2].set_ylabel("count"); ax[2].set_title("Connectivity"); ax[2].set_ylim(0, 70)
ax[2].legend(fontsize=8)

fig.suptitle("USPS, 1001 images", fontsize=12)
plt.tight_layout(); plt.show()


### Against the numbers quoted in section 6.2


In [ ]:
best = DF_USPS.groupby("model").clustering_error.min()
rows = [("best clustering error, Ours", "0.24 or 0.25", f"{best['Ours']:.2f}"),
        ("best clustering error, Dong etal", "0.25 or 0.24", f"{best['Dong etal']:.2f}")]
for mo, comp, disc in [("Ours", 3, 0), ("Dong etal", 35, 22)]:
    r, _ = pick(DF_USPS, W_USPS, mo, 6.0)
    rows += [(f"~6 edges/node, {mo}: components", str(comp), str(int(r.n_components))),
             (f"~6 edges/node, {mo}: disconnected nodes", str(disc),
              str(int(r.n_disconnected_nodes))),
             (f"~6 edges/node, {mo}: density reached", "6.0", f"{r.density:.1f}")]
print(pd.DataFrame(rows, columns=["quantity", "paper", "here"]).to_string(index=False))


### Timing at 10 edges per node


In [ ]:
m = Xu.shape[0]
for name, model, col, mx, paper in [("Algorithm 1", "log", "Ours", MAXIT_LOG, "5 s, 218 it"),
                                    ("Algorithm 2", "dong", "Dong etal", MAXIT_DONG, "60 s, 2043 it")]:
    r, _ = pick(DF_USPS, W_USPS, col, 10.0)
    for eps in (1e-4, EPS):
        t0 = time.perf_counter()
        if model == "log":
            w, _, it = gl.fbf_primal_dual(Z_USPS, 1.0, r.param, m, eps=eps,
                                          max_iter=mx, warn=False)
        else:
            w, _, it = gl.fbf_primal_dual_dong(Z_USPS, r.param, float(m), m, eps=eps,
                                               max_iter=mx, warn=False)
        print(f"{name} at eps={eps:.0e}: {time.perf_counter()-t0:6.1f} s, {it:6d} iterations, "
              f"density {mt.density(gl.vec_to_W(w, m)):5.1f}"
              + (f"    paper: {paper}" if eps == 1e-4 else ""))


### USPS graphs at ~6 edges per node, nodes coloured by digit


In [ ]:
from sklearn.manifold import TSNE
COORD_U = TSNE(n_components=2, init="pca", random_state=0,
               perplexity=30).fit_transform(Xu)
DIGIT_U = np.array([0 if c == "10" else int(c) for c in yu])

fig, ax = plt.subplots(1, 3, figsize=(18, 6.2))
for a, mo in zip(ax, ["Ours", "Dong etal", "k-NN"]):
    r, w = pick(DF_USPS, W_USPS, mo, 6.0)
    sc = draw(a, w, COORD_U, DIGIT_U, cmap="tab10", vmin=-0.5, vmax=9.5,
              title=f"{mo}   density {r.density:.1f}\n{int(r.n_components)} components, "
                    f"{int(r.n_disconnected_nodes)} isolated nodes")
cb = fig.colorbar(sc, ax=ax, fraction=0.013, ticks=range(10))
cb.set_label("digit")
plt.show()


### The same graphs, nodes coloured by their weighted degree $(Sw)_i$


In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(18, 6.2))
for a, mo in zip(ax, ["Ours", "Dong etal", "k-NN"]):
    r, w = pick(DF_USPS, W_USPS, mo, 6.0)
    deg = gl.degrees(w, Xu.shape[0])
    deg = deg / deg.max()
    sc = draw(a, w, COORD_U, deg, cmap="viridis", vmin=0, vmax=1,
              title=f"{mo}   min degree {deg.min():.4f}"
                    + ("   <- zeros = isolated nodes" if deg.min() < 1e-9 else ""))
cb = fig.colorbar(sc, ax=ax, fraction=0.013)
cb.set_label("degree (normalised)")
plt.show()


## MNIST "1" vs "2"


In [ ]:
PROPS    = [(40, 160), (100, 100), (160, 40)]
N_DRAWS  = 40
LP_RUNS  = 20
B_MNIST  = np.logspace(np.log10(0.02), np.log10(1.5), 10)
A_MNIST  = np.logspace(np.log10(0.03), np.log10(2.0), 10)
K_MNIST  = [1, 2, 3, 4, 5, 6, 8, 10]

ONES = MNIST_X[MNIST_y == "1"]
TWOS = MNIST_X[MNIST_y == "2"]

def mnist_sample(n1, n2, seed):
    r = np.random.default_rng(seed)
    X = np.vstack([ONES[r.choice(len(ONES), n1, replace=False)],
                   TWOS[r.choice(len(TWOS), n2, replace=False)]])
    return X, np.array(["1"] * n1 + ["2"] * n2)

def one_draw(Xm, ym, key, d):
    df, _, _ = sweep(Xm, ym, label_rate=0.10, cluster=False, n_runs=LP_RUNS, rng=d,
                     betas=B_MNIST, alphas=A_MNIST, ks=K_MNIST, verbose=False, n_jobs=1)
    df["prop"], df["draw"] = key, d
    return df

t0 = time.perf_counter()
jobs = [(*mnist_sample(n1, n2, 1000 * n1 + d), f"{n1} ones / {n2} twos", d)
        for n1, n2 in PROPS for d in range(N_DRAWS)]
print(f"{len(jobs)} draws of m = {jobs[0][0].shape[0]}")
DF_MNIST = pd.concat(Parallel(n_jobs=-1, backend="loky", verbose=5)(
    delayed(one_draw)(*j) for j in jobs), ignore_index=True)
print(f"total {time.perf_counter()-t0:.0f}s,  {len(DF_MNIST)} graphs")


### Figure 2


In [ ]:
AGG = (DF_MNIST.groupby(["prop", "model", "param"], as_index=False)
       [["density", "lp_miss", "lp_unclass_rate", "n_components"]].mean())

fig, ax = plt.subplots(1, 3, figsize=(17, 4.6), sharey=True)
for a, (n1, n2) in zip(ax, PROPS):
    key = f"{n1} ones / {n2} twos"
    curve(a, AGG[AGG["prop"] == key], "lp_miss", dash="lp_unclass_rate", xmax=18)
    a.set_title(key, fontsize=11); a.set_ylim(0, 0.4)
ax[0].set_ylabel("classification error"); ax[0].legend(fontsize=9)
fig.suptitle('MNIST "1" vs "2", averaged over 40 draws', fontsize=12)
plt.tight_layout(); plt.show()


### Miss rate at sparse graphs (4 edges per node)


In [ ]:
rows = []
for key in AGG["prop"].unique():
    for mo in ["Ours", "Dong etal", "k-NN"]:
        s = AGG[(AGG["prop"] == key) & (AGG.model == mo)]
        r = s.iloc[int((s.density - 4).abs().argmin())]
        rows.append({"proportion": key, "model": mo, "density": round(r.density, 1),
                     "miss rate": round(r.lp_miss, 3),
                     "unclassifiable": round(r.lp_unclass_rate, 3)})
print(pd.DataFrame(rows).to_string(index=False))


### One MNIST graph, nodes coloured by the label-propagation value


In [ ]:
Xm, ym = mnist_sample(40, 160, 1000 * 40 + 0)
dfm, zm, wm = sweep(Xm, ym, label_rate=0.10, cluster=False, n_runs=LP_RUNS,
                    betas=B_MNIST, alphas=A_MNIST, ks=K_MNIST, keep_w=True,
                    verbose=False)
COORD_M = TSNE(n_components=2, init="pca", random_state=0,
               perplexity=25).fit_transform(Xm)
IS_TWO = (ym == "2").astype(float)

from scipy.linalg import solve as _solve
def harmonic(W, labels, rate=0.10, seed=0):
    m = W.shape[0]
    r = np.random.default_rng(seed)
    lab = r.choice(m, max(int(round(rate * m)), 1), replace=False)
    is_lab = np.zeros(m, bool); is_lab[lab] = True
    f = np.full(m, np.nan); f[is_lab] = (labels[is_lab] == "2").astype(float)
    L = np.diag(W.sum(1)) - W
    from scipy.sparse.csgraph import connected_components
    _, comp = connected_components((np.abs(W) > 1e-6 * np.abs(W).max()).astype(np.int8),
                                   directed=False)
    for c in np.unique(comp):
        nd = np.flatnonzero(comp == c)
        lc, uc = nd[is_lab[nd]], nd[~is_lab[nd]]
        if uc.size == 0 or lc.size == 0:
            continue
        f[uc] = np.clip(np.linalg.lstsq(L[np.ix_(uc, uc)],
                                        W[np.ix_(uc, lc)] @ f[lc], rcond=None)[0], 0, 1)
    return f

fig, ax = plt.subplots(2, 3, figsize=(18, 11))
for col, mo in enumerate(["Ours", "Dong etal", "k-NN"]):
    r, w = pick(dfm, wm, mo, 4.0)
    W = gl.vec_to_W(w, Xm.shape[0])
    sc1 = draw(ax[0, col], w, COORD_M, IS_TWO, cmap="coolwarm", vmin=0, vmax=1,
               title=f"{mo}   density {r.density:.1f}, {int(r.n_components)} components\n"
                     f"true class (blue = 1, red = 2)", s=34)
    f = harmonic(W, ym)
    sc2 = draw(ax[1, col], w, COORD_M, np.nan_to_num(f, nan=0.5), cmap="coolwarm",
               vmin=0, vmax=1, s=34,
               title=f"label-propagation value, "
                     f"{int(np.isnan(f).sum())} nodes unreachable")
fig.colorbar(sc2, ax=ax, fraction=0.013, label="value at the node")
plt.show()


## COIL-20 (appendix C.1)


In [ ]:
N_REP_COIL = 5
B_COIL = np.logspace(np.log10(0.02), np.log10(1.0), 11)
A_COIL = np.logspace(np.log10(0.03), np.log10(1.5), 11)
K_COIL = [2, 3, 4, 5, 6, 8, 10, 13, 16, 20, 25]
COIL_MEANS = 3.0 * (60.0 / 3.0) ** (np.arange(20) / 19.0)

def coil_sample(seed):
    r = np.random.default_rng(seed)
    cls = np.unique(COIL_y)
    sizes = np.clip(r.poisson(COIL_MEANS), 1, 72)
    idx = np.concatenate([r.choice(np.flatnonzero(COIL_y == c), s, replace=False)
                          for c, s in zip(cls, sizes)])
    return COIL_X[idx], COIL_y[idx], sizes

_, _, s0 = coil_sample(0)
print("one instance, class sizes:", s0, " m =", s0.sum())

t0 = time.perf_counter()
coil_rows = []
for rep in range(N_REP_COIL):
    Xc, yc, _ = coil_sample(rep)
    df, _, _ = sweep(Xc, yc, label_rate=0.50, cluster=True, n_runs=N_RUNS, rng=rep,
                     betas=B_COIL, alphas=A_COIL, ks=K_COIL, verbose=False)
    df["rep"] = rep
    coil_rows.append(df)
    print(f"  rep {rep}: m = {Xc.shape[0]}, {time.perf_counter()-t0:6.1f}s")
DF_COIL = pd.concat(coil_rows, ignore_index=True)
print(f"total {time.perf_counter()-t0:.0f}s")


### Figure 6


In [ ]:
AGG_C = (DF_COIL.groupby(["model", "param"], as_index=False)
         [["density", "clustering_error", "lp_miss", "lp_unclass_rate",
           "n_components", "n_disconnected_nodes"]].mean())

fig, ax = plt.subplots(1, 3, figsize=(17, 4.4))
curve(ax[0], AGG_C, "clustering_error", xmax=25)
ax[0].set_ylabel("clustering error"); ax[0].set_title("Spectral clustering")
ax[0].legend(fontsize=9)

curve(ax[1], AGG_C, "lp_miss", dash="lp_unclass_rate", xmax=25)
ax[1].set_ylabel("classification error"); ax[1].set_title("Label propagation")

curve(ax[2], AGG_C, "n_components", xmax=25)
sub = AGG_C[(AGG_C.model == "Dong etal") & (AGG_C.density <= 25)].sort_values("density")
ax[2].plot(sub.density, sub.n_disconnected_nodes, "--", color=COLORS["Dong etal"],
           label="# disconnected (Dong etal)")
ax[2].axhline(20, color="k", lw=1, ls=":", label="number of classes")
ax[2].set_ylabel("count"); ax[2].set_title("Graph connectivity")
ax[2].legend(fontsize=8)

fig.suptitle(f"COIL-20, averaged over {N_REP_COIL} samples", fontsize=12)
plt.tight_layout(); plt.show()


### One COIL-20 graph, nodes coloured by object class


In [ ]:
Xc, yc, _ = coil_sample(0)
dfc, zc, wc = sweep(Xc, yc, label_rate=0.50, cluster=False, n_runs=20,
                    betas=B_COIL, alphas=A_COIL, ks=K_COIL, keep_w=True, verbose=False)
COORD_C = TSNE(n_components=2, init="pca", random_state=0,
               perplexity=25).fit_transform(Xc)

fig, ax = plt.subplots(1, 3, figsize=(18, 6.2))
for a, mo in zip(ax, ["Ours", "Dong etal", "k-NN"]):
    r, w = pick(dfc, wc, mo, 6.0)
    sc = draw(a, w, COORD_C, yc.astype(int), cmap="tab20", vmin=0.5, vmax=20.5, s=22,
              title=f"{mo}   density {r.density:.1f}\n{int(r.n_components)} components, "
                    f"{int(r.n_disconnected_nodes)} isolated nodes")
cb = fig.colorbar(sc, ax=ax, fraction=0.013, ticks=range(1, 21))
cb.set_label("object")
plt.show()


## Summary against the paper's claims


In [ ]:
claims = []
b = DF_USPS.groupby("model").clustering_error.min()
claims.append(["USPS", "best clustering error, Ours vs Dong etal",
               "0.24 vs 0.25", f"{b['Ours']:.2f} vs {b['Dong etal']:.2f}"])
for mo, c, d in [("Ours", 3, 0), ("Dong etal", 35, 22)]:
    r, _ = pick(DF_USPS, W_USPS, mo, 6.0)
    claims.append(["USPS", f"~6 edges/node, {mo}: components / isolated",
                   f"{c} / {d}", f"{int(r.n_components)} / {int(r.n_disconnected_nodes)}"])
claims.append(["USPS", "Ours and k-NN have no isolated nodes at any density",
               "yes",
               "yes" if DF_USPS[DF_USPS.model != "Dong etal"].n_disconnected_nodes.max() == 0
                     else "no"])
for key in AGG["prop"].unique():
    s = AGG[(AGG["prop"] == key)]
    o = s[s.model == "Ours"]; d = s[s.model == "Dong etal"]
    o = o.iloc[int((o.density - 4).abs().argmin())]
    d = d.iloc[int((d.density - 4).abs().argmin())]
    claims.append(["MNIST", f"{key}: miss rate at ~4 edges/node, Ours vs Dong etal",
                   "Ours lower", f"{o.lp_miss:.3f} vs {d.lp_miss:.3f}"])
cd = AGG_C[(AGG_C.model == "Dong etal") & (AGG_C.density <= 25)]
co_ = AGG_C[(AGG_C.model == "Ours") & (AGG_C.density <= 25)]
claims.append(["COIL-20", "max isolated nodes, Dong etal vs Ours", "Dong etal only",
               f"{cd.n_disconnected_nodes.max():.0f} vs {co_.n_disconnected_nodes.max():.0f}"])
print(pd.DataFrame(claims, columns=["dataset", "claim", "paper", "here"]).to_string(index=False))
